![Header Image](../assets/header_image.png "Header Image")

# Devoir Optionnel 5 : Visualisation de Données Capteurs avec ROS 2

Bienvenue dans ce tutoriel sur la visualisation de données capteurs en **ROS 2** !

> **Pré-requis :** Avoir complété le notebook `3_introduction_to_ros2_fr.ipynb`.
> Sélectionnez le kernel **"Python 3.8 (ROS 2 Foxy)"**
> (**Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**).

Dans ce devoir, vous allez

- **comprendre les différences entre `rosbag` (ROS 1) et `ros2 bag` (ROS 2)**
- **générer et publier des données capteurs synthétiques** : LIDAR, IMU, odométrie
- **visualiser un nuage de points LIDAR** en 2D et 3D avec `matplotlib`
- **visualiser une trajectoire d'odométrie** calculée à partir de messages `Odometry`
- **enregistrer et rejouer des données** avec `ros2 bag`

# ROS 1 vs ROS 2 : Données Capteurs et Bags

| Aspect | ROS 1 | ROS 2 |
|--------|--------|--------|
| **Enregistrement** | `rosbag record -a` | `ros2 bag record -a` |
| **Lecture** | `rosbag play fichier.bag` | `ros2 bag play dossier_bag/` |
| **Format de fichier** | `.bag` (format propriétaire) | `.db3` (SQLite3) ou `.mcap` |
| **Visualisation 3D** | Zethus + RosBridge + websocket | `rviz2` ou matplotlib directement |
| **Inspection** | `rosbag info fichier.bag` | `ros2 bag info dossier_bag/` |

> **Note importante :** Les fichiers `.bag` de ROS 1 ne sont **pas directement lisibles** par `ros2 bag`.
> Un outil de conversion `rosbag2_bag_v2` existe, mais n'est pas installé dans ce conteneur.
> Dans ce notebook, nous **générons des données capteurs synthétiques** pour illustrer les concepts.

# Configurer l'environnement

In [ ]:
!source /opt/ros/foxy/setup.bash

In [ ]:
import sys
sys.path.insert(0, '/opt/ros/foxy/lib/python3.8/site-packages/')

import platform
print("Python utilisé :", platform.python_version())
assert platform.python_version_tuple()[1] == '8', \
    "ERREUR : mauvais kernel ! Sélectionnez 'Python 3.8 (ROS 2 Foxy)' dans Kernel >> Change Kernel."
print("Kernel OK — Python 3.8 confirmé.")

In [ ]:
import subprocess, sys, importlib

# Installer ipywidgets pour Python 3.8 si absent
try:
    import ipywidgets
    print("ipywidgets déjà disponible.")
except ModuleNotFoundError:
    print("Installation de ipywidgets (première fois, ~10 s)...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets'])
    # pip installe dans ~/.local/lib/python3.8/site-packages/ (hors sys.path du kernel)
    # On détecte l'emplacement réel et on l'ajoute à sys.path
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', 'ipywidgets'],
                       capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith('Location:'):
            loc = line.split(':', 1)[1].strip()
            if loc not in sys.path:
                sys.path.insert(0, loc)
            break
    importlib.invalidate_caches()
    print("ipywidgets installé et chemin ajouté à sys.path.")

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan, Imu, PointCloud2
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Quaternion

import threading
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Bibliothèques importées avec succès !")

# Initialiser ROS 2

In [ ]:
rclpy.init()
node = rclpy.create_node('capteurs_node')
spin_thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
spin_thread.start()
print(f"Nœud '{node.get_name()}' créé et thread ROS 2 démarré.")

# Partie 1 : LIDAR — Scan 2D avec LaserScan

## Comprendre le message LaserScan

Le message `sensor_msgs/LaserScan` représente un scan LIDAR 2D :

```
std_msgs/Header header
float32 angle_min       # angle de départ du scan (radians)
float32 angle_max       # angle de fin du scan (radians)
float32 angle_increment # résolution angulaire (radians/step)
float32 time_increment  # temps entre deux mesures (secondes)
float32 scan_time       # durée d'un scan complet (secondes)
float32 range_min       # distance minimale (mètres)
float32 range_max       # distance maximale (mètres)
float32[] ranges        # tableau de distances
float32[] intensities   # tableau d'intensités (optionnel)
```

Nous allons simuler un LIDAR à 360° avec une résolution de 1° (360 points par scan).

In [ ]:
def generer_scan_lidar(t, bruit=0.05):
    """
    Génère un scan LIDAR synthétique simulant un couloir rectangulaire
    avec un obstacle mobile (autre robot ou piéton).
    """
    N = 360  # 360 points par scan, un par degré
    angles = np.linspace(-math.pi, math.pi, N)
    ranges = np.zeros(N)

    # Murs du couloir (rectangle 8m x 4m)
    L, W = 4.0, 2.0  # demi-longueur et demi-largeur
    for i, a in enumerate(angles):
        ca, sa = math.cos(a), math.sin(a)
        # Distance aux 4 murs
        d_list = []
        if ca > 0.001:  d_list.append( L / ca)
        if ca < -0.001: d_list.append(-L / ca)
        if sa > 0.001:  d_list.append( W / sa)
        if sa < -0.001: d_list.append(-W / sa)
        d_mur = min(d_list) if d_list else 10.0

        # Obstacle circulaire mobile
        ox = 1.5 * math.cos(0.5 * t)
        oy = 0.8 * math.sin(0.5 * t)
        # Distance du rayon à l'obstacle (rayon = 0.3 m)
        d_ray_obs = ox * math.cos(a) + oy * math.sin(a)
        perp2 = (ox - d_ray_obs * ca)**2 + (oy - d_ray_obs * sa)**2
        if perp2 < 0.3**2 and d_ray_obs > 0:
            d_obs = d_ray_obs - math.sqrt(max(0.0, 0.3**2 - perp2))
        else:
            d_obs = 999.0

        d = min(d_mur, d_obs)
        ranges[i] = max(0.1, d + np.random.normal(0, bruit))

    return angles, ranges.astype(np.float32)

# Test de la fonction
angles_test, ranges_test = generer_scan_lidar(t=0.0)
print(f"Scan généré : {len(ranges_test)} points, distance min={ranges_test.min():.2f}m, max={ranges_test.max():.2f}m")

# Publier et Visualiser le LIDAR en Temps Réel

In [ ]:
pub_lidar = node.create_publisher(LaserScan, '/scan', 10)

dernier_scan = {'angles': None, 'ranges': None}
stop_lidar = threading.Event()

def boucle_lidar():
    t0 = time.time()
    while not stop_lidar.is_set():
        t = time.time() - t0
        angles, ranges = generer_scan_lidar(t)

        msg = LaserScan()
        msg.angle_min       = float(-math.pi)
        msg.angle_max       = float(math.pi)
        msg.angle_increment = float(2 * math.pi / len(ranges))
        msg.range_min       = 0.1
        msg.range_max       = 10.0
        msg.ranges          = ranges.tolist()
        pub_lidar.publish(msg)

        dernier_scan['angles'] = angles
        dernier_scan['ranges'] = ranges

        time.sleep(0.1)  # 10 Hz

lidar_thread = threading.Thread(target=boucle_lidar, daemon=True)
lidar_thread.start()

# Visualisation interactive du LIDAR
out_lidar = widgets.Output()
display(out_lidar)
stop_lidar_viz = threading.Event()

def viz_lidar():
    while not stop_lidar_viz.is_set():
        if dernier_scan['angles'] is None:
            time.sleep(0.1)
            continue
        a = dernier_scan['angles']
        r = dernier_scan['ranges']
        x = r * np.cos(a)
        y = r * np.sin(a)

        with out_lidar:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(x, y, s=2, c=r, cmap='viridis', vmin=0, vmax=5)
            ax.plot(0, 0, 'r^', markersize=10, label='Robot')
            ax.set_xlim(-5, 5)
            ax.set_ylim(-5, 5)
            ax.set_aspect('equal')
            ax.set_title('LIDAR 2D — /scan (ROS 2)\n(bleu=proche, jaune=loin)')
            ax.set_xlabel('x (m)')
            ax.set_ylabel('y (m)')
            ax.legend(loc='upper right')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        time.sleep(0.3)

lidar_viz_thread = threading.Thread(target=viz_lidar, daemon=True)
lidar_viz_thread.start()

print("LIDAR démarré sur /scan — visualisation en temps réel (5s d'observation)...")
time.sleep(5)
stop_lidar_viz.set()
print("Observation terminée.")

# Snapshot statique du dernier scan LIDAR

Affichage avec la distance codée en couleur sur un graphique polaire.

In [ ]:
a = dernier_scan['angles']
r = dernier_scan['ranges']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graphique polaire
ax_polar = plt.subplot(121, projection='polar')
sc = ax_polar.scatter(a, r, c=r, cmap='plasma', s=3, vmin=0, vmax=5)
ax_polar.set_title('Vue polaire du scan LIDAR', pad=15)
plt.colorbar(sc, ax=ax_polar, label='Distance (m)', pad=0.1)

# Vue cartésienne avec obstacle identifié
ax_cart = plt.subplot(122)
x_scan = r * np.cos(a)
y_scan = r * np.sin(a)
# Colorer les points proches de l'obstacle (< 2m)
mask_obstacle = r < 2.0
ax_cart.scatter(x_scan[~mask_obstacle], y_scan[~mask_obstacle], s=3, c='steelblue', alpha=0.5, label='Murs')
ax_cart.scatter(x_scan[mask_obstacle],  y_scan[mask_obstacle],  s=8, c='red',       alpha=0.9, label='Obstacle (<2m)')
ax_cart.plot(0, 0, 'g^', markersize=12, label='Robot')
ax_cart.set_title('Vue cartésienne avec détection d\'obstacle')
ax_cart.set_xlabel('x (m)')
ax_cart.set_ylabel('y (m)')
ax_cart.set_aspect('equal')
ax_cart.legend()
ax_cart.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Obstacle détecté : {mask_obstacle.sum()} points à moins de 2m")

# Partie 2 : Odométrie — Suivi de Trajectoire

## Le message Odometry

Le message `nav_msgs/Odometry` contient la pose et la vitesse estimées du robot :

```
std_msgs/Header header
string child_frame_id
geometry_msgs/PoseWithCovariance pose     # Position + orientation + incertitude
geometry_msgs/TwistWithCovariance twist   # Vitesse linéaire + angulaire + incertitude
```

Nous allons simuler un robot naviguant sur un circuit ovale et récupérer sa trajectoire.

In [ ]:
pub_odom = node.create_publisher(Odometry, '/odom', 10)

# Stocker les messages reçus
odom_data = {'x': [], 'y': [], 'theta': [], 't': [], 'v_lin': [], 'v_ang': []}

def callback_odom(msg):
    odom_data['x'].append(msg.pose.pose.position.x)
    odom_data['y'].append(msg.pose.pose.position.y)
    odom_data['v_lin'].append(msg.twist.twist.linear.x)
    odom_data['v_ang'].append(msg.twist.twist.angular.z)

sub_odom = node.create_subscription(Odometry, '/odom', callback_odom, 10)

def euler_vers_quaternion(yaw):
    q = Quaternion()
    q.w = math.cos(yaw / 2)
    q.z = math.sin(yaw / 2)
    return q

# Simuler un circuit ovale
print("Publication d'odométrie sur un circuit ovale (5 secondes)...")
t0 = time.time()
while time.time() - t0 < 5.0:
    t = time.time() - t0
    # Circuit ovale paramétrique
    omega = 0.8  # rad/s
    a_ellipse, b_ellipse = 3.0, 1.5  # demi-axes
    phi = omega * t
    x_r   =  a_ellipse * math.cos(phi)
    y_r   =  b_ellipse * math.sin(phi)
    theta_r = math.atan2(-a_ellipse * math.sin(phi), b_ellipse * math.cos(phi))
    v_lin = omega * math.sqrt((a_ellipse * math.sin(phi))**2 + (b_ellipse * math.cos(phi))**2)
    v_ang = omega

    msg = Odometry()
    msg.pose.pose.position.x = x_r + np.random.normal(0, 0.01)
    msg.pose.pose.position.y = y_r + np.random.normal(0, 0.01)
    msg.pose.pose.position.z = 0.0
    msg.pose.pose.orientation = euler_vers_quaternion(theta_r)
    msg.twist.twist.linear.x  = v_lin
    msg.twist.twist.angular.z = v_ang
    pub_odom.publish(msg)

    time.sleep(0.02)  # 50 Hz

print(f"{len(odom_data['x'])} messages d'odométrie reçus via callback.")

# Visualiser la Trajectoire et les Vitesses

In [ ]:
if len(odom_data['x']) < 10:
    print("Pas assez de données — ré-exécutez la cellule précédente.")
else:
    x_arr = np.array(odom_data['x'])
    y_arr = np.array(odom_data['y'])
    n = len(x_arr)
    coul = np.arange(n)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Trajectoire
    sc = axes[0].scatter(x_arr, y_arr, c=coul, cmap='viridis', s=8)
    axes[0].plot(x_arr[0], y_arr[0], 'go', markersize=10, label='Départ')
    axes[0].plot(x_arr[-1], y_arr[-1], 'rs', markersize=10, label='Arrivée')
    axes[0].set_title('Trajectoire du robot — /odom (ROS 2)')
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    axes[0].set_aspect('equal')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.colorbar(sc, ax=axes[0], label='Pas de temps')

    # Vitesses
    t_ax = np.linspace(0, 5, n)
    axes[1].plot(t_ax, odom_data['v_lin'], label='v_lin (m/s)',  color='steelblue')
    axes[1].plot(t_ax, odom_data['v_ang'], label='v_ang (rad/s)', color='darkorange')
    axes[1].set_title('Vitesses de déplacement')
    axes[1].set_xlabel('Temps (s)')
    axes[1].set_ylabel('Vitesse')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Partie 3 : IMU — Capteur Inertiel

## Le message IMU

Le message `sensor_msgs/Imu` contient les mesures d'une centrale inertielle :

```
std_msgs/Header header
geometry_msgs/Quaternion orientation          # Orientation (quaternion)
float64[9] orientation_covariance
geometry_msgs/Vector3 angular_velocity        # Gyroscope (rad/s)
float64[9] angular_velocity_covariance
geometry_msgs/Vector3 linear_acceleration     # Accéléromètre (m/s²)
float64[9] linear_acceleration_covariance
```

Nous simulons les mesures d'un véhicule prenant un virage à droite.

In [ ]:
pub_imu = node.create_publisher(Imu, '/imu/data', 10)

imu_data = {'t': [], 'acc_x': [], 'acc_y': [], 'acc_z': [], 'gyro_z': []}

def callback_imu(msg):
    imu_data['acc_x'].append(msg.linear_acceleration.x)
    imu_data['acc_y'].append(msg.linear_acceleration.y)
    imu_data['acc_z'].append(msg.linear_acceleration.z)
    imu_data['gyro_z'].append(msg.angular_velocity.z)

sub_imu = node.create_subscription(Imu, '/imu/data', callback_imu, 10)

print("Publication de données IMU (4 secondes)...")
t0 = time.time()
while time.time() - t0 < 4.0:
    t = time.time() - t0

    # Simulation : ligne droite (t<1s), virage à droite (1s<t<3s), ligne droite (t>3s)
    if t < 1.0 or t > 3.0:
        acc_x = 0.5 + np.random.normal(0, 0.1)   # Accélération en avant
        acc_y = 0.0 + np.random.normal(0, 0.05)  # Pas de force latérale
        gyro_z = 0.0 + np.random.normal(0, 0.02)
    else:
        acc_x = 0.3 + np.random.normal(0, 0.1)   # Décélération légère en virage
        acc_y = -2.5 + np.random.normal(0, 0.1)  # Force centrifuge vers la droite
        gyro_z = -0.8 + np.random.normal(0, 0.02) # Rotation à droite

    msg = Imu()
    msg.linear_acceleration.x = acc_x
    msg.linear_acceleration.y = acc_y
    msg.linear_acceleration.z = 9.81 + np.random.normal(0, 0.05)  # Gravité
    msg.angular_velocity.z    = gyro_z
    msg.orientation = euler_vers_quaternion(gyro_z * t)
    pub_imu.publish(msg)

    imu_data['t'].append(t)
    time.sleep(0.01)  # 100 Hz

print(f"{len(imu_data['t'])} échantillons IMU publiés et reçus.")

In [ ]:
t_imu = imu_data['t']
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(t_imu, imu_data['acc_x'], label='acc_x (avant/arrière)', color='steelblue')
axes[0].plot(t_imu, imu_data['acc_y'], label='acc_y (gauche/droite)', color='darkorange')
axes[0].plot(t_imu, imu_data['acc_z'], label='acc_z (vertical/gravité)', color='green', alpha=0.5)
axes[0].axvspan(1.0, 3.0, alpha=0.1, color='red', label='Phase virage')
axes[0].set_ylabel('Accélération (m/s²)')
axes[0].set_title('IMU — Accéléromètre et Gyroscope (/imu/data)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_imu, imu_data['gyro_z'], label='gyro_z (lacet)', color='purple')
axes[1].axvspan(1.0, 3.0, alpha=0.1, color='red', label='Phase virage')
axes[1].set_xlabel('Temps (s)')
axes[1].set_ylabel('Vitesse angulaire (rad/s)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Zone rouge = phase de virage à droite (t=1s à t=3s)")

# Partie 4 : Utiliser ros2 bag — Enregistrement et Lecture

## Enregistrer un bag depuis le terminal

Ouvrez un terminal (**Fichier >> Nouveau >> Terminal**) et exécutez :

```bash
source /opt/ros/foxy/setup.bash

# Enregistrer tous les topics dans un bag
ros2 bag record -a -o /home/jovyan/tp-va/section_1_introduction_and_tools/mon_bag
```

Pendant l'enregistrement, exécutez les cellules de publication ci-dessus.
Appuyez sur **Ctrl+C** dans le terminal pour arrêter l'enregistrement.

## Inspecter le bag enregistré

```bash
ros2 bag info /home/jovyan/tp-va/section_1_introduction_and_tools/mon_bag/
```

Vous verrez quelque chose comme :
```
Files:             mon_bag_0.db3
Bag size:          245.3 KiB
Storage id:        sqlite3
Duration:          5.123s
Start:             ...
End:               ...
Messages:          1234
Topic information: Topic: /scan | Type: sensor_msgs/msg/LaserScan | Count: 51
                   Topic: /odom | Type: nav_msgs/msg/Odometry | Count: 256
                   Topic: /imu/data | Type: sensor_msgs/msg/Imu | Count: 400
```

## Rejouer le bag

```bash
# Rejouer en boucle
ros2 bag play /home/jovyan/tp-va/section_1_introduction_and_tools/mon_bag/ --loop

# Rejouer à vitesse réduite (0.5x)
ros2 bag play /home/jovyan/tp-va/section_1_introduction_and_tools/mon_bag/ -r 0.5

# Pendant la lecture, dans un autre terminal :
ros2 topic list
ros2 topic echo /scan
```

> **Différence clé ROS 1 vs ROS 2 :** En ROS 1, `rosbag play` crée un nœud maître si nécessaire.
> En ROS 2, `ros2 bag play` publie directement sur le middleware DDS — aucun master requis.

# Arrêter le LIDAR et libérer les ressources

In [ ]:
stop_lidar.set()
stop_lidar_viz.set()
time.sleep(0.3)

node.destroy_node()
rclpy.shutdown()

print("Tous les threads arrêtés.")
print("Nœud ROS 2 détruit — ressources DDS libérées.")

# Résumé

- Vous avez compris les **différences entre `rosbag` (ROS 1) et `ros2 bag` (ROS 2)** : format SQLite3, commandes différentes, pas besoin de master.
- Vous avez **généré et publié des données LIDAR synthétiques** (`LaserScan`) simulant un environnement avec obstacle mobile.
- Vous avez **visualisé le scan LIDAR** en vue polaire et cartésienne avec détection d'obstacles.
- Vous avez **simulé une trajectoire d'odométrie** et visualisé la trajectoire et les vitesses.
- Vous avez **simulé des données IMU** et identifié les phases de virage dans les signaux accéléromètre/gyroscope.
- Vous avez appris à utiliser `ros2 bag record` et `ros2 bag play` depuis le terminal.